# 4	Development and Validation of an Integrated Prediction Model

## 4.1	Positive data augmentation strategy: 

task 31: By analysis the 14-3-3 protein homology in evolution tree, we found that 14-3-3 is quite conservative until XX animal (Figure X). 

task 32: Utilizing this approach, we selected 5 different species (XX,XX) where the 14-3-3 protein sequences are conserved. To ensure adequate diversity, we also included different isoforms from these species. (sampling from uniref90?, then make sure by MSA, the sequence have a similary 14-3-3 sites)

In [1]:
# 原始数据中有的蛋白上有5-1个位点，增强的数据应该把同源蛋白的数量填充到最多五个， 比如如果某个蛋白上有2个位点，那么我们只增强三个位点，使总蛋白数是5
# 选1.uniref_90 2. aligment 3. at the positive have the same motif （比如 rxxs or rxxt） 4.blast similary 大致相同， 5. 在特定的种群列表里面 6。 距离最远的4个, # 增强的数据位点，在用以前的predictor测试一下，保证增强数据在他们的短距离的predictor中是推荐的位点

In [2]:
### Download all homologous sequences for a given protein UniProt ID within a specified list of species.

In [3]:
# install Blast+
# https://blast.ncbi.nlm.nih.gov/doc/blast-help/downloadblastdata.html#downloadblastdata

In [4]:
# https://ftp.uniprot.org/pub/databases/uniprot/current_release/uniref/uniref90/
# download uniref90
# build db uniref90 for blast search and retrieve
# seems too big, maybe I should filter the species first

In [ ]:
# 0. still need to filter the species first, than build the db
# 1. blast search once for each uniprot, 
# 2. select the uniprot Id for each species
# 3. for each uniprot id, find the isoform for it

In [5]:
import requests
import xml.etree.ElementTree as ET

def get_isoforms(uniprot_id):
    """Fetch isoforms for a given UniProt ID."""
    base_url = f"https://www.ebi.ac.uk/proteins/api/proteins/{uniprot_id}/isoforms"
    print(base_url)
    response = requests.get(base_url)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch isoforms for {uniprot_id} from UniProt.")

    root = ET.fromstring(response.content)
    namespace = {'uniprot': 'http://uniprot.org/uniprot'}
    
    isoforms = []
    for isoform in root.findall('.//uniprot:isoform', namespaces=namespace):
        isoform_id = isoform.find('uniprot:id', namespaces=namespace).text
        isoforms.append(isoform_id)
    
    return list(set(isoforms + [uniprot_id]))

# # Example usage
# uniprot_id = "Q06413"
# isoforms = get_isoforms(uniprot_id)
# print(isoforms)

In [6]:
import os
import requests
from Bio.Blast.Applications import NcbiblastpCommandline
from Bio.Blast import NCBIXML
import re

def fetch_sequence(uniprot_id):
    """Fetch the protein sequence from UniProt given a UniProt ID."""
    sequence_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(sequence_url)
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve sequence for {uniprot_id}")
    
    sequence = response.text  # return the full FASTA format for use with BLAST
    print(f"Fetched sequence for {uniprot_id}:\n{sequence}")
    return sequence

def perform_blast(sequence, db, target_species, output_file='blast_result.xml'):
    """Perform a BLAST search using local BLAST and return the best hit for the target species."""
    with open("query.fasta", "w") as f:
        f.write(sequence)
    
    # Specify the full path to the blastp executable
    blastp_path = "/Users/newuser/ncbi-blast-2.16.0+/bin/blastp"  # Update this with the actual path to blastp on your system

    blastp_cline = NcbiblastpCommandline(cmd=blastp_path, query="query.fasta", db=db, evalue=0.001, outfmt=5, out=output_file)
    
    # Run BLASTP and capture stdout and stderr
    stdout, stderr = blastp_cline()
    # print(f"stdout: {stdout}")
    # print(f"stderr: {stderr}")
    
    with open(output_file) as result_handle:
        blast_records = NCBIXML.read(result_handle)
    
    for alignment in blast_records.alignments:
        hit_id = alignment.hit_id
        hit_def = alignment.hit_def
        print(f"Checking hit: {hit_def}")  # Debug output
        uniprot_id, species = get_species_uniprot_id(hit_id, hit_def)
        if species == target_species:
            return alignment
    
    return None

def get_species_uniprot_id(hit_id, hit_def):
    """
    Extract UniProt ID from hit_id and species from the BLAST hit definition.
    
    Example hit_id: 'sp|P12345.1|'
    Example hit_def: 'RecName: Full=Tissue-type plasminogen activator; OS=Homo sapiens...'
    """
    # Extract UniProt ID from hit_id
    uniprot_match = re.match(r'sp\|(\w+\.\d+)\|', hit_id)
    uniprot_id = uniprot_match.group(1) if uniprot_match else None
    
    # Extract species name from hit_def
    # Regular expression to match content within brackets
    pattern = r'\[(.*?)\]'
    
    # Search for the specified pattern in the input string
    match = re.search(pattern, hit_def)
    
    # Extract the content within the brackets
    species = match.group(1) if match else None
    
    # Debug print
    print(f"Extracted UniProt ID: {uniprot_id}, Species: {species}")
    
    return uniprot_id, species

def reciprocal_blast(uniprot_id, target_species, species_db, original_db):
    """Perform RBH to find orthologs or return the best hit using local BLAST."""
    # Fetch sequence for the given UniProt ID
    query_sequence = fetch_sequence(uniprot_id)
    
    # 1. BLAST query sequence against species target proteome
    target_hit = perform_blast(query_sequence, species_db, target_species)
    if target_hit:
        print(f"Initial BLAST target hit: {target_hit.hit_id}")
    else:
        raise Exception(f"No target hit found in first BLAST search for {target_species}.")
    
    # Extract UniProt ID from target_hit definition
    target_uniprot_id, _ = get_species_uniprot_id(target_hit.hit_id, target_hit.hit_def)
    
    # Fetch sequence for the target hit
    target_sequence = fetch_sequence(target_uniprot_id)
    
    # 2. BLAST target hit sequence back to the original species proteome
    original_hit = perform_blast(target_sequence, original_db, "Homo sapiens")
    if original_hit:
        print(f"Reciprocal BLAST original hit: {original_hit.hit_id}")
    else:
        raise Exception("No original hit found in second BLAST search.")
    
    # Get UniProt ID from original hit title
    original_best_hit_id, _ = get_species_uniprot_id(original_hit.hit_id, original_hit.hit_def)
    
    # Check if the original query UniProt ID matches the best hit ID
    if uniprot_id == original_best_hit_id:
        return target_uniprot_id
    else:
        return target_uniprot_id  # Return the best hit even if it's not reciprocal

# Example usage
uniprot_id = "Q06413-3"
target_species = "Mus musculus"  # Change this to the desired species you are looking for
species_db = "/Users/newuser/blastdb/uniref90"  # Absolute path to the species BLAST database
original_db = "/Users/newuser/blastdb/uniref90"  # Absolute path to the original species BLAST database (e.g., human proteome)

try:
    ortholog_id = reciprocal_blast(uniprot_id, target_species, species_db, original_db)
    if ortholog_id:
        print(f"Ortholog or best hit found from species {target_species}: {ortholog_id}")
    else:
        print("No hit found.")
except Exception as e:
    print(f"Error: {str(e)}")

/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


Fetched sequence for Q06413-3:
>sp|Q06413-3|MEF2C_HUMAN Isoform 3 of Myocyte-specific enhancer factor 2C OS=Homo sapiens OX=9606 GN=MEF2C
MGRKKIQITRIMDERNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSTNKLFQYAST
DMDKVLLKYTEYNEPHESRTNSDIVETLRKKGLNGCDSPDPDADDSVGHSPESEDKYRKI
NEDIDLMISRQRLCAVPPPNFEMPVSIPVSSHNSLVYSNPVSSLGNPNLLPLAHPSLQRN
SMSPGVTHRPPSAGNTGGLMGGDLTSGAGTSAGNGYGNPRNSPGLLVSPGNLNKNMQAKS
PPPMNLGMNNRKPDLRVLIPPGSKNTMPSVSEDVDLLLNQRINNSQSAQSLATPVVSVAT
PTLPGQGMGGYPSAISTTYGTEYSLSSADLSSLSGFNTASALHLGSVTGWQQQHLHNMPP
SALSQLGDRTTTPSRYPQHTRHEAGRSPVDSLSSCSSSYDGSDREDHRNEFHSPIGLTRP
SPDERESPSVKRMRLSEGWAT

Error: Non-zero return code 3 from '/Users/newuser/ncbi-blast-2.16.0+/bin/blastp -out blast_result.xml -outfmt 5 -query query.fasta -db /Users/newuser/blastdb/uniref90 -evalue 0.001', message 'BLAST engine error: Database memory map file error'


In [22]:
import requests
from Bio import SeqIO
from io import StringIO

def download_all_homologs(uniprot_id, species_list):
    # Get the isoforms of the provided UniProt ID
    isoforms = get_isoforms(uniprot_id)
    print(f"Number of isoforms found for {uniprot_id}: {len(isoforms)}")
    
    all_homologs = []

    # Get homologous sequences for each isoform and the original UniProt ID
    for isoform in isoforms:
        for species_id in species_list:
            print(f"Fetching homologous sequences for isoform {isoform} in {species_id}")
            homolog = reciprocal_blast(isoform, species_id)
            print(f"Number of homologous sequences for isoform {isoform}: {len(homolog)}")
            all_homologs.append(homolog)

    return all_homologs

# Example usage
uniprot_id = "Q06413"
species_list = ["10090"]  # Human (Homo sapiens), Mouse (Mus musculus)
all_homologs = download_all_homologs(uniprot_id, species_list)

print(set(all_homologs))

https://www.ebi.ac.uk/proteins/api/proteins/Q06413/isoforms
Number of isoforms found for Q06413: 7
Fetching homologous sequences for isoform Q06413-5 in 10090
MGRKKIQITRIMDERNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSTNKLFQYASTDMDKVLLKYTEYNEPHESRTNSDIVETLRKKGLNGCDSPDPDADDSALNKKENKGCESPDPDSSYALTPRTEEKYKKINEEFDNMIKSHKIPAVPPPNFEMPVSIPVSSHNSLVYSNPVSSLGNPNLLPLAHPSLQRNSMSPGVTHRPPSAGNTGGLMGGDLTSGAGTSAGNGYGNPRNSPGLLVSPGNLNKNMQAKSPPPMNLGMNNRKPDLRVLIPPGSKNTMPSVNQRINNSQSAQSLATPVVSVATPTLPGQGMGGYPSAISTTYGTEYSLSSADLSSLSGFNTASALHLGSVTGWQQQHLHNMPPSALSQLGACTSTHLSQSSNLSLPSTQSLNIKSEPVSPPRDRTTTPSRYPQHTRHEAGRSPVDSLSSCSSSYDGSDREDHRNEFHSPIGLTRPSPDERESPSVKRMRLSEGWAT


/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/Bio/Blast/NCBIWWW.py:281: BiopythonWarning: BLAST request RM3Y1PWZ016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


MGRKKIQITRIMDERNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSTNKLFQYASTDMDKVLLKYTEYNEPHESRTNSDIVEALNKKENKGSESPDPDSSYALTPRTEEKYKKINEEFDNMIKSHKIPAVPPPSLEMPVTIPVSSHNSLVYSNPVSTLGNPNLLPLAHPSLQRNSMSPGVTHRPPSAGNTGGLMGGDLTSGAGTSAGNGYGNPRNSPGLLVSPGNLNKNIQAKSPPPMNLGMNNRKPDLRVLIPPGSKNTMPSVSEDVDLLLNQRINNSQSAQSLATPVVSVATPTLPGQGMGGYPSAISTTYGTEYSLSSADLSSLSGFNTASALHLGSVTGWQQQHLHNMPPSALSQLGACTSTHLSQSSNLSLPSTQSLSIKSEPVSPPRDRTTTPSRYPQHTTRHEAGRSPVDSLSSCSSSYDGSDREDHRNEFHSPIGLTRPSPDERESPSVKRMRLSEGWAT


/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/Bio/Blast/NCBIWWW.py:281: BiopythonWarning: BLAST request RM4PFT9B016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


Number of homologous sequences for isoform Q06413-5: 8
Fetching homologous sequences for isoform Q06413-3 in 10090
MGRKKIQITRIMDERNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSTNKLFQYASTDMDKVLLKYTEYNEPHESRTNSDIVETLRKKGLNGCDSPDPDADDSVGHSPESEDKYRKINEDIDLMISRQRLCAVPPPNFEMPVSIPVSSHNSLVYSNPVSSLGNPNLLPLAHPSLQRNSMSPGVTHRPPSAGNTGGLMGGDLTSGAGTSAGNGYGNPRNSPGLLVSPGNLNKNMQAKSPPPMNLGMNNRKPDLRVLIPPGSKNTMPSVSEDVDLLLNQRINNSQSAQSLATPVVSVATPTLPGQGMGGYPSAISTTYGTEYSLSSADLSSLSGFNTASALHLGSVTGWQQQHLHNMPPSALSQLGDRTTTPSRYPQHTRHEAGRSPVDSLSSCSSSYDGSDREDHRNEFHSPIGLTRPSPDERESPSVKRMRLSEGWAT


...

MGRKKIQITRIMDERNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSTNKLFQYASTDMDKVLLKYTEYNEPHESRTNSDIVEALNKKENKGSESPDPDSSYALTPRTEEKYKKINEEFDNMIKSHKIPAVPPPSLEMPVTIPVSSHNSLVYSNPVSTLGNPNLLPLAHPSLQRNSMSPGVTHRPPSAGNTGGLMGGDLTSGAGTSAGNGYGNPRNSPGLLVSPGNLNKNIQAKSPPPMNLGMNNRKPDLRVLIPPGSKNTMPSVSEDVDLLLNQRINNSQSAQSLATPVVSVATPTLPGQGMGGYPSAISTTYGTEYSLSSADLSSLSGFNTASALHLGSVTGWQQQHLHNMPPSALSQLGACTSTHLSQSSNLSLPSTQSLSIKSEPVSPPRDRTTTPSRYPQHTTRHEAGRSPVDSLSSCSSSYDGSDREDHRNEFHSPIGLTRPSPDERESPSVKRMRLSEGWAT


/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/Bio/Blast/NCBIWWW.py:281: BiopythonWarning: BLAST request RM6AB1U1016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


Number of homologous sequences for isoform Q06413-3: 8
Fetching homologous sequences for isoform Q06413-1 in 10090
MGRKKIQITRIMDERNRQVTFTKRKFGLMKKAYELSVLCDCEIALIIFNSTNKLFQYASTDMDKVLLKYTEYNEPHESRTNSDIVETLRKKGLNGCDSPDPDADDSVGHSPESEDKYRKINEDIDLMISRQRLCAVPPPNFEMPVSIPVSSHNSLVYSNPVSSLGNPNLLPLAHPSLQRNSMSPGVTHRPPSAGNTGGLMGGDLTSGAGTSAGNGYGNPRNSPGLLVSPGNLNKNMQAKSPPPMNLGMNNRKPDLRVLIPPGSKNTMPSVSEDVDLLLNQRINNSQSAQSLATPVVSVATPTLPGQGMGGYPSAISTTYGTEYSLSSADLSSLSGFNTASALHLGSVTGWQQQHLHNMPPSALSQLGACTSTHLSQSSNLSLPSTQSLNIKSEPVSPPRDRTTTPSRYPQHTRHEAGRSPVDSLSSCSSSYDGSDREDHRNEFHSPIGLTRPSPDERESPSVKRMRLSEGWAT


/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/Bio/Blast/NCBIWWW.py:281: BiopythonWarning: BLAST request RM77342W016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
###    Ensure that the multiple sequence alignment (MSA) has serine (S) or threonine (T) at the corresponding positions.

In [1]:
import subprocess
from Bio import AlignIO

def run_clustal_omega(in_file, out_file):
    # Running clustal omega using subprocess
    result = subprocess.run(
        ["clustalo", "-i", in_file, "-o", out_file, "--force", "--verbose"],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print("Error running Clustal Omega:")
        print(result.stderr)
        raise RuntimeError(f"Clustal Omega failed with return code {result.returncode}")

def check_motif_in_alignment(alignment_file, position, motif):
    alignment = AlignIO.read(alignment_file, "fasta")
    for record in alignment:
        sequence = str(record.seq)
        if sequence[position:position+len(motif)] != motif:
            print(f"Motif not found at specified position in sequence {record.id}")
            return False
    return True

# Input sequences in FASTA format
input_sequences = """
>seq1
ACGTACGTACGT
>seq2
ACGTACGACGT
>seq3
CGTACGTACGT
"""

with open("input.fasta", "w") as file:
    file.write(input_sequences)

# Run Clustal Omega to align sequences
run_clustal_omega("input.fasta", "aligned.fasta")

# Specify the position and motif
position = 2
motif = "GTAC"

# Check for the motif at the specified position
if check_motif_in_alignment("aligned.fasta", position, motif):
    print("Motif found at the specified position in all sequences.")
else:
    print("Motif not found in all sequences at the specified position.")

FileNotFoundError: [Errno 2] No such file or directory: 'clustalo'

In [ ]:
### Use MMseqs2 to cluster the remaining sequences and select a seed sequence from each cluster.
### Use a prediction tool to ensure the target site can bind 14-3-3 proteins.

## 4.2	Negative data sampling strategy

task 33: we propose a negative data sampling strategy to augment the negative data. We first calculate the protein condensation,  IDR and phosphorylation feature（我到底应不应该用condensation 做负采样啊用multiple scale feature 做负采样，也许可以用IDR feature， 或者用embedding做负采样）score using human proteome, then bin and draw the distribution of those multiple scale features. When compare whole proteome distribution with the distribution of 14-3-3 binding proteins, we could see indeed the dataset we have is biased (Figure 4A). (Add some details about the distribution looks like).

task 34: we sampled data points from all the S and T site from human proteome to make the negative sample distribution close to the whole human proteome distribution. We exclude sampling from protein which have been reported interaction with 14-3-3, and exclude the site with high score site predicted using previous 14-3-3 predictor. 

task 35: Then, to reduce the sequence redundancy of proteins and avoid model overfitting, the CD-HIT program was used with the sequence identity threshold of 30%. After sample, we could see the distribution of condensation, IDR, phosphorylation now follow the same distribution of the whole protein and there is no significant different when tested by XX test

## 4.3	Feature selection 

task 36: modeling direct from only PLM embedding, we first compare using embeddings from either ESM2 or PTM-mamba to represent the whole protein or the residue embeddings, 

task 37: Then we tested modeling from multi-scale biology features plus onehot encoding for the -7 to +7 motif, the result show that multi-scale biology features also can be used to predict the 14-3-3 binding when compare with only use onehot encoded motif feature. 

task 38: We then test using both PLM and biology features, the results show that using both PLM and biology features shows improved results compared with using only PLM or multi-scale biology features. 

task 39: feature selection using mRMR. We mainly test how many features are optimized for our model by using different number of features to fit the model and compare the model performance.

task 40: We also test feature selection using another method Boruta, it shows that XX feature give us the best results.

task 41: feature importance and its contribution were
further analyzed to find which feature was more valuable for
the model performance after feature selection 

## 4.4	Prediction performance with different classifiers

task 42: To test the validity of the optimal feature set in different classifiers, three common classifiers were used to predict 14-3-3 sites: random forest (RF), XGBoost (XGB), SVM and ANN are used. 

## 4.5	Model architecture and optimization 

task 43: fine-tuned certain hyperparameters (Table X) of the model

## 4.6 Feature Importance analysis

task 44: we conducted a thorough examination of feature importance using SHAP (SHapley Additive exPlanations) values

task 45: ablation experiments where we systematically removed features to observe changes in model performance.

## 4.7	Performance evaluation and comparison with existing methods, Validation on independent test sets

task 46: Independent datset test with 14-3-3 pred and 14-3-3 site-finder, compare with our results.

# 5	Evaluation of the clinically relevant variant and their possible regulation by 14-3-3 and condensation. 

task 47: download and curate prediction data from clinvar

task 48: prediction

task 49: figure

## 5.1 active learning for 14-3-3 model updating

task: model update

# 6	Further understanding the relationship of 14-3-3 and condensation

task: We further understand how 14-3-3 might affect condensation. 
We first select a condensation dataset which has been provided can form condensation and separate it into two part base on the predicted binding with 14-3-3. Then we characterized the difference between the 14-3-3 and non-14-3-3 binding condensation protein 

task: We also compare the characterization of 14-3-3 binding and condensation proteins with 14-3-3 binding but no-condensation proteins.  Using those two datasets, we can compare what is the difference between them. We are looking for 1. The number of binding sites in the protein. 2. Distance between binding sites in the protein. 3. Location of binding sites related to condensation key motifs. 4. Distance between binding site and sticker/loop region in condensation. 5. Binding features of the interaction sites. The results show that XXX

# 7	Website, write an agentic model for prediction of 14-3-3 protein